In [ ]:
import json
from os import listdir as ls
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, ANALYSIS_TYPES
from emu_renewal.plotting import plot_analysis_specific_post
from emu_renewal.utils import get_countries_by_continent, split_list_into_segments, get_analysis_paths

set_matplotlib_formats("svg")

Need to make this handle both mob_exp and scale_floor for all the scaled analyses.

In [ ]:
import numpy as np
import arviz as az
from matplotlib import pyplot as plt
import pycountry
from emu_renewal.constants import MOB_SOURCE_COLOURS

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(countries) / n_cols))
height = min(2.0 + n_rows * 2.0, 13.0)
fig, axes = plt.subplots(n_rows, n_cols, figsize=[12, height])
flat_axes = axes.ravel()

param_name = "scale_floor"
for c, iso3 in enumerate(countries):
    a_paths = analysis_paths[iso3]
    ax = flat_axes[c]
    scale_paths = {k: v for k, v in a_paths.items() if k != "no_mob" and "fb_" not in k}
    if not scale_paths:
        ax.set_axis_off()
    for a in scale_paths:
        a_path = a_paths[a]
        idata = az.from_netcdf(a_path / "idata_filtered.nc")
        colour = MOB_SOURCE_COLOURS[a]
        az.plot_density(idata, ax=ax, hdi_prob=0.99, var_names=param_name, shade=0.2, colors=colour)
    ax.set_title(pycountry.countries.lookup(iso3).name)
fig.tight_layout()

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/inc_with_pol.json", "r"))
analysis_paths = get_analysis_paths(["49427755", "49507800", "49508160"], all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
for cont, cont_countries in countries_by_cont.items():
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"## {cont_name}"))
    for countries in split_list_into_segments(cont_countries, 16):
        analyses = analysis_paths[country]
        fig = plot_analysis_specific_post(job_path, countries, "oxcgrt", "scale_floor", 4)
        display(fig)